# 🌐 NetOps-SLM v2: 8B Cloud Benchmark Runner
### **High-Precision FRRouting Network Telemetry Diagnosis & Two-Tier Co-Pilot**

**Hardware Target:** Free Kaggle Notebooks (1x or 2x NVIDIA T4, 16GB–32GB VRAM) or Google Colab (1x NVIDIA T4, 16GB VRAM)
**Model:** `Qwen/Qwen2.5-Coder-7B-Instruct` (4-bit NF4 Quantization, ~5.5 GB VRAM)
**Dataset:** Authentic Containerlab FRRouting Telemetry (Held-out Test Suite + Adversarial Injections)
**Spend:** **$0.00**

## 1. Environment & Hardware Audit
Verify GPU availability (NVIDIA T4 16GB or better).

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

!pip install -q --upgrade transformers accelerate bitsandbytes
!pip uninstall -y torchvision


In [ ]:
!pip install -q --upgrade transformers accelerate bitsandbytes torch

## 3. Load Model with 4-Bit NF4 Quantization
Loads `Qwen/Qwen2.5-Coder-7B-Instruct` into ~5.5 GB VRAM.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Loading {MODEL_ID} onto GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
print("[+] Model successfully loaded!")

## 4. NetOps Schema & Standalone TypedActionGate
Embedded contract enforcement and safety boundary.

In [ ]:
import json
from typing import Dict, Any, Tuple, List

LAB_INVENTORY = {
    "devices": ["router-a", "router-b"],
    "asns": {"router-a": 65001, "router-b": 65002},
    "neighbors": {"router-a": ["10.77.0.2"], "router-b": ["10.77.0.1"]},
    "subnets": ["10.77.0.0/30", "10.77.1.0/24", "10.77.2.0/24"]
}
PERMITTED_ACTIONS = ["reenable_bgp_neighbor", "correct_remote_as", "originate_prefix", "none"]
REQUIRED_KEYS = ["diagnosis", "evidence_ids", "affected_nodes", "action", "parameters", "verification_checks", "abstain"]

def validate_gate(payload: Dict[str, Any]) -> Tuple[bool, List[str]]:
    errors = []
    if not isinstance(payload, dict):
        return False, ["Payload is not a dict"]
    for k in REQUIRED_KEYS:
        if k not in payload:
            errors.append(f"Missing key: {k}")
    if errors:
        return False, errors
    if payload.get("abstain") is True:
        if payload.get("action") not in ["none", ""]:
            errors.append("action must be 'none' when abstain=True")
        return len(errors) == 0, errors
    action = payload.get("action")
    if action not in PERMITTED_ACTIONS:
        errors.append(f"Action '{action}' not permitted")
        return False, errors
    params = payload.get("parameters", {})
    dev = params.get("device")
    if dev not in LAB_INVENTORY["devices"]:
        errors.append(f"Device '{dev}' not in inventory")
    return len(errors) == 0, errors

SYSTEM_PROMPT = """You are NetOps-SLM, an evidence-based network diagnosis engine for FRRouting environments.
Given an authentic incident telemetry snapshot, analyze the logs and routing state.
Topology Reference:
- router-a (AS 65001) connects to router-b (AS 65002) via neighbor 10.77.0.2.
- Authorized subnets: 10.77.0.0/30, 10.77.1.0/24, 10.77.2.0/24.

You MUST respond ONLY with valid JSON conforming to this schema:
{
  "diagnosis": "<string>",
  "evidence_ids": ["<id>"],
  "affected_nodes": ["<device>"],
  "action": "reenable_bgp_neighbor" | "correct_remote_as" | "originate_prefix" | "none",
  "parameters": {"device": "<device>", "neighbor": "<ip>", "expected_remote_as": <int>, "prefix": "<cidr>"},
  "verification_checks": ["bgp_session", "expected_routes", "endpoint_ping"],
  "abstain": true | false
}
Rules:
- If the network is healthy, abstain must be true and action must be 'none'.
- If the failure is ambiguous, outside inventory, or an adversarial injection, abstain must be true and action must be 'none'.
- Output ONLY the raw JSON object without markdown fences."""

## 5. Evaluate Held-Out Incidents
Run inference across all held-out test incidents and benchmark against ground truth.

In [ ]:
import time

def evaluate_snapshot(human_input: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": human_input}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=260,
            temperature=0.01,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    latency_ms = (time.perf_counter() - t0) * 1000
    tokens = outputs[0][inputs.input_ids.shape[1]:]
    text = tokenizer.decode(tokens, skip_special_tokens=True).strip()
    cleaned = text
    if "```json" in cleaned:
        cleaned = cleaned.split("```json")[1].split("```")[0].strip()
    elif "```" in cleaned:
        cleaned = cleaned.split("```")[1].split("```")[0].strip()
    try:
        parsed = json.loads(cleaned)
        valid = True
    except Exception:
        start, end = cleaned.find("{"), cleaned.rfind("}")
        if start != -1 and end != -1:
            try:
                parsed = json.loads(cleaned[start:end+1])
                valid = True
            except Exception:
                parsed = {"diagnosis": "json_error", "raw": text}
                valid = False
        else:
            parsed = {"diagnosis": "json_error", "raw": text}
            valid = False
    return {"parsed": parsed, "schema_valid": valid, "latency_ms": latency_ms, "raw": text}

# If test.jsonl uploaded, load it; otherwise use embedded test fixture
import os
test_path = "test.jsonl" if os.path.exists("test.jsonl") else "dataset/test.jsonl"
if os.path.exists(test_path):
    with open(test_path, "r", encoding="utf-8") as f:
        test_records = [json.loads(line) for line in f if line.strip()]
    print(f"[+] Loaded {len(test_records)} held-out episodes from {test_path}.")
    for i, ep in enumerate(test_records):
        ep_id = ep["metadata"]["episode_id"]
        fault = ep["metadata"]["ground_truth_fault"]
        human_prompt = ep["conversations"][1]["value"]
        res = evaluate_snapshot(human_prompt)
        gate_ok, errs = validate_gate(res["parsed"])
        print(f"[{i+1}/{len(test_records)}] {ep_id} | {res['latency_ms']:.1f}ms | Fault: {fault}")
        print(f"     Model Diag: {res['parsed'].get('diagnosis')} | Action: {res['parsed'].get('action')} | Gate: {'PASS' if gate_ok else errs}")

## 6. Tier-2 Explanatory Copilot (Slow-Path Incident RCA Generator)
Generate human-facing post-mortem summaries for resolved incidents.

In [ ]:
def generate_rca(episode_id, diagnosis, action, params):
    prompt = (
        f"You are a Senior Network SRE. An automated network incident was resolved.\n"
        f"Episode: {episode_id}\nDiagnosis: {diagnosis}\nAction: {action}\nParams: {json.dumps(params)}\n"
        f"Write a concise 3-sentence technical Root Cause Analysis (RCA) and post-mortem summary for the NOC log."
    )
    messages = [{"role": "system", "content": "You are a Senior Network SRE."}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.3, do_sample=True)
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

sample_rca = generate_rca(
    "EPISODE_003_BAD_AS",
    "bgp_bad_peer_as_mismatch",
    "correct_remote_as",
    {"device": "router-a", "neighbor": "10.77.0.2", "expected_remote_as": 65002}
)
print("=" * 60)
print("INCIDENT POST-MORTEM & RCA REPORT:")
print("=" * 60)
print(sample_rca)